In [3]:
# import
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models


In [4]:
#preprocessing
datapath = "../../../desktop/quant/hist/aaplIntra.csv"
df = pd.read_csv(datapath)


In [5]:
df

,Dates,Open,Close,High,Low,Volume,Number Ticks
0,7/1/25 9:30,206.665,206.915,207.08,206.600,1035492,1433
1,7/1/25 9:30,206.910,206.710,206.92,206.500,119487,721
2,7/1/25 9:30,206.730,206.810,206.95,206.695,95679,603
3,7/1/25 9:30,206.840,207.200,207.22,206.790,164543,948
4,7/1/25 9:30,207.200,207.115,207.24,206.980,123276,626
...,...,...,...,...,...,...,...
281104,##########,273.900,273.670,274.60,273.470,95766657,2970
281105,##########,273.670,273.670,273.67,273.670,0,1
281106,12/19/25 15:59,273.690,273.900,273.91,273.600,556667,1933
281107,12/19/25 15:59,273.900,273.670,274.60,273.470,95766657,2970


In [ ]:
feature_cols = ["Open", "High", "Low", "Close", "Volume", "Number Ticks"]
X_all = df[feature_cols].values.astype("float32")

# next bar price label from close
close = df["Close"].values.astype("float32")
y_all = np.roll(close, -1)

# drop last bar
X_all = X_all[:-1]
y_all = y_all[:-1]

# 4) turn into sequences of length T
T = 30
Xs = []
ys = []
for i in range(len(X_all) - T + 1):
    Xs.append(X_all[i:i+T])
    ys.append(y_all[i+T-1])
Xs = np.stack(Xs)        # shape (N, T, F)
ys = np.array(ys)        # shape (N,)

# 5) train validation split
split = int(0.8 * len(Xs))
X_train, X_val = Xs[:split], Xs[split:]
y_train, y_val = ys[:split], ys[split:]


In [ ]:
timesteps = X_train.shape[1]   # T
features = X_train.shape[2]    # F

model = tf.keras.Sequential([
    tf.keras.layers.LSTM(128, return_sequences=True, input_shape=(timesteps, features), dropout=0.2),
    tf.keras.layers.LSTM(64, return_sequences=False, dropout=0.2),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1)  #linear activation for regression
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_10 (LSTM)                  │ (None, 30, 128)        │        69,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 120,641 (471.25 KB)

 Trainable params: 120,641 (471.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 109s 30ms/step - loss: 11373.5645 - mae: 75.9431 - val_loss: 1588.4912 - val_mae: 39.5534
Epoch 2/10
3512/3514 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 1532.4244 - mae: 31.4305

In [8]:
model.save('models/lstm.keras')

In [15]:
print(tf.config.list_physical_devices('GPU'))

[]
